In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,15),
#                     dates < datetime(2026,9,6)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,20),
                     dates < datetime(2026,10,6)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,15),
#                     dates < datetime(2026,10,6)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260820T002003_V202610201927C_0648200021.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20261005T093003_V202610211201C_0650050504.fits'),
 153)

In [18]:
file = files[-50]
print(file)

with fits.open(file) as hdul:
    header = hdul[0].header
    data = hdul[0].data

header['CROTA'] += 0.25

/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260924T063003_V202610202208C_0649240503.fits


In [19]:
data_, view = reproject(data, header, kind='synoptic', correct_dr=True, car_rot=2)

In [21]:
fig, ax = show_data(data_, view, cmap='seismic', vmin=-50, vmax=50)

In [27]:
plt.figure(figsize=(10,8))
plt.imshow(data_[600:,:150])

In [3]:
Q = []
dates = []

for file in files[80:]:
    with fits.open(file) as hdul:
        header = hdul[0].header
        data = hdul[0].data
    header['CROTA'] += 0.25

    data_, view = reproject(data, header, kind='synoptic', correct_dr=True, car_rot=2)
    #show_data(data_, view, cmap='seismic', vmin=-50, vmax=50, to_folder='png')

    Q += [data_[600:,:150]]
    dates += [datetime.fromisoformat(header['DATE-OBS'])]

Q = np.array(Q)
dates = np.array(dates)

In [4]:
plt.figure(figsize=(10,8))
plt.imshow(Q[15], cmap='seismic', vmin=-50, vmax=50, origin='lower')

In [5]:
plt.figure(figsize=(10,8))
plt.imshow(Q[30], cmap='seismic', vmin=-50, vmax=50, origin='lower')

In [6]:
plt.figure(figsize=(10,8))
plt.imshow(Q[45], cmap='seismic', vmin=-50, vmax=50, origin='lower')

In [7]:
plt.figure(figsize=(10,8))
plt.imshow(Q[65], cmap='seismic', vmin=-50, vmax=50, origin='lower')

In [40]:
from scipy.ndimage import map_coordinates
dt = np.array([(date - dates[0]).total_seconds() / 3600 for date in dates])

q = np.nanmean(Q[:,:,20:140], axis=-1).T
nx, ny = q.shape

#xi, yi = np.mgrid[:nx,:ny]
#yi_ = np.expand_dims(dt / np.max(dt) * (ny - 1), 0) * np.ones((nx, 1))

q_ = map_coordinates(q, (xi, yi_))

fig, ax = plt.subplots(figsize=(10,8))
ax.imshow(q_, cmap='seismic', vmin=-50, vmax=50, origin='lower', aspect='auto')

#ax.set_xticklabels(np.floor(dt))

plt.tight_layout()

In [65]:
yi = np.arange(0,ny) / ny#(ny - 1)
dt_ = dt / np.max(dt)

p = np.searchsorted(dt_, yi, side='right')

a = yi - dt_[(p - 1).clip(0)]
b = dt_[p.clip(0,ny - 1)] - yi

q_ = q[:,(p - 1).clip(0)] * b / (a + b + 1e-8) + a / (a + b + 1e-8) * q[:, p.clip(0,ny - 1)]

fig, ax = plt.subplots(figsize=(10,8))
ax.imshow(q_, cmap='seismic', vmin=-50, vmax=50, origin='lower', aspect='auto')

#ax.set_xticklabels(np.floor(dt))

plt.tight_layout()

In [59]:
dt_

array([0.        , 0.01600006, 0.03200008, 0.04800003, 0.06400005,
       0.08000009, 0.09600015, 0.10400007, 0.11200005, 0.12000004,
       0.14666677, 0.15200002, 0.16000005, 0.19466671, 0.2026667 ,
       0.20800003, 0.21600005, 0.22400007, 0.23200011, 0.24000007,
       0.248     , 0.27200006, 0.28000009, 0.2880001 , 0.29600001,
       0.33600002, 0.34399998, 0.35200002, 0.36000007, 0.40000001,
       0.41600005, 0.42400008, 0.43200007, 0.49599998, 0.50400005,
       0.52800004, 0.53600005, 0.544     , 0.55200003, 0.56      ,
       0.56800006, 0.58400004, 0.59199995, 0.60000001, 0.608     ,
       0.61600008, 0.62400003, 0.63200007, 0.63999999, 0.65599998,
       0.66400004, 0.67200005, 0.68000009, 0.72      , 0.72800002,
       0.7359999 , 0.74399997, 0.78399987, 0.79199995, 0.79999997,
       0.80800001, 0.84799999, 0.856     , 0.86399999, 0.87200001,
       0.91199997, 0.92000002, 0.92800001, 0.93599998, 0.97600003,
       0.98399996, 0.99199997, 1.        ])

In [60]:
yi

array([0.        , 0.01388889, 0.02777778, 0.04166667, 0.05555556,
       0.06944444, 0.08333333, 0.09722222, 0.11111111, 0.125     ,
       0.13888889, 0.15277778, 0.16666667, 0.18055556, 0.19444444,
       0.20833333, 0.22222222, 0.23611111, 0.25      , 0.26388889,
       0.27777778, 0.29166667, 0.30555556, 0.31944444, 0.33333333,
       0.34722222, 0.36111111, 0.375     , 0.38888889, 0.40277778,
       0.41666667, 0.43055556, 0.44444444, 0.45833333, 0.47222222,
       0.48611111, 0.5       , 0.51388889, 0.52777778, 0.54166667,
       0.55555556, 0.56944444, 0.58333333, 0.59722222, 0.61111111,
       0.625     , 0.63888889, 0.65277778, 0.66666667, 0.68055556,
       0.69444444, 0.70833333, 0.72222222, 0.73611111, 0.75      ,
       0.76388889, 0.77777778, 0.79166667, 0.80555556, 0.81944444,
       0.83333333, 0.84722222, 0.86111111, 0.875     , 0.88888889,
       0.90277778, 0.91666667, 0.93055556, 0.94444444, 0.95833333,
       0.97222222, 0.98611111, 1.        ])

In [61]:
p

array([ 1,  1,  2,  3,  4,  5,  6,  7,  8, 10, 10, 12, 13, 13, 13, 16, 17,
       19, 21, 21, 22, 24, 25, 25, 25, 27, 29, 29, 29, 30, 31, 32, 33, 33,
       33, 33, 34, 35, 35, 37, 39, 41, 41, 43, 45, 47, 48, 49, 51, 53, 53,
       53, 54, 56, 57, 57, 57, 58, 60, 61, 61, 61, 63, 65, 65, 65, 66, 68,
       69, 69, 69, 71, 73])